# Test of voice command classification using deep learning

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import sounddevice as sd
import numpy as np
from torch.utils.data import DataLoader
from torchaudio.datasets import SPEECHCOMMANDS

In [ ]:
# ==========================================
# 1. CONFIGURATION & DATASET LOADING
# ==========================================
BATCH_SIZE = 64
SAMPLE_RATE = 16000
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# Automatically downloads Google Speech Commands v0.02 (~2GB) if not present
class SubsetSC(SPEECHCOMMANDS):
    def __init__(self, subset: str = None):
        super().__init__("./data", download=True)
        # Standard train/val/test splits provided by the dataset metadata
        def load_list(filename):
            filepath = os.path.join(self._path, filename)
            with open(filepath) as fileobj:
                return [os.path.join(self._path, line.strip()) for line in fileobj]

        if subset == "validation":
            self._walker = load_list("validation_list.txt")
        elif subset == "testing":
            self._walker = load_list("testing_list.txt")
        elif subset == "training":
            excludes = set(load_list("validation_list.txt") + load_list("testing_list.txt"))
            self._walker = [w for w in self._walker if w not in excludes]

train_set = SubsetSC("training")
test_set = SubsetSC("testing")

# Target command labels (subset of 35 words or all depending on choice)
labels = sorted(list(set(datapoint[2] for datapoint in train_set)))
word_to_idx = {word: i for i, word in enumerate(labels)}

def collate_fn(batch):
    tensors, targets = [], []
    for waveform, sample_rate, label, speaker_id, utterance_number in batch:
        # Ensure uniform length (1 second = 16,000 samples)
        if waveform.shape[1] < SAMPLE_RATE:
            pad = SAMPLE_RATE - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :SAMPLE_RATE]
        
        tensors.append(waveform)
        targets.append(word_to_idx[label])
        
    return torch.stack(tensors), torch.tensor(targets)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# ==========================================
# 2. FEATURE EXTRACTION & MODEL ARCHITECTURE
# ==========================================
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_mels=64
).to(DEVICE)

class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = AudioCNN(num_classes=len(labels)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
# ==========================================
# 3. TRAINING LOOP
# ==========================================
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for waveforms, targets in train_loader:
        waveforms, targets = waveforms.to(DEVICE), targets.to(DEVICE)
        
        # Extract Mel-spectrograms on the fly
        spectrograms = mel_transform(waveforms)
        
        optimizer.zero_grad()
        outputs = model(spectrograms)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss / len(train_loader):.4f}")

In [ ]:
# ==========================================
# 4. LIVE MICROPHONE INFERENCE SANDBOX
# ==========================================
def listen_and_predict():
    model.eval()
    duration = 1.0  # 1-second sample window
    print("\nListening for a command ('yes', 'no', 'stop', 'go', etc.)...")
    
    audio_np = sd.rec(int(SAMPLE_RATE * duration), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
    sd.wait()
    
    waveform = torch.from_numpy(np.squeeze(audio_np)).unsqueeze(0).to(DEVICE)
    spec = mel_transform(waveform).unsqueeze(0) # Add batch dim
    
    with torch.no_grad():
        output = model(spec)
        pred_idx = torch.argmax(output, dim=1).item()
        predicted_label = labels[pred_idx]
        
    print(f"Detected Command: **{predicted_label.upper()}**")

# Uncomment to test live with your microphone:
# listen_and_predict()